In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import re
import os
import psutil
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")
print(f"Pandas version : {pd.__version__}")
print(f"Numpy version  : {np.__version__}")

Libraries loaded!
Pandas version : 2.3.2
Numpy version  : 2.2.3


In [46]:
def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem     = process.memory_info().rss / 1024**2
    return round(mem, 1)

print("Memory helper ready!")

Memory helper ready!


In [47]:
optimized_dtypes = {
    'id'                : 'int32',
    'vote_average'      : 'float32',
    'vote_count'        : 'int32',
    'revenue'           : 'float32',
    'runtime'           : 'float32',
    'budget'            : 'float32',
    'popularity'        : 'float32',
    'adult'             : 'bool',
    'original_language' : 'category',
    'status'            : 'category',
}

useful_columns = [
    'id', 'title', 'overview', 'genres',
    'keywords', 'vote_average', 'vote_count',
    'popularity', 'release_date', 'original_language',
    'poster_path', 'status', 'adult',
    'runtime', 'imdb_id'
]

print(f"Memory BEFORE: {get_memory_usage()} MB")

df = pd.read_csv(
    'data/TMDB_movie_dataset_v11.csv',
    usecols=useful_columns,
    dtype=optimized_dtypes
)

print(f"Memory AFTER : {get_memory_usage()} MB")
print(f"Shape        : {df.shape}")

Memory BEFORE: 17.8 MB
Memory AFTER : 936.8 MB
Shape        : (1385673, 15)


In [48]:
print(f"Shape before filtering: {df.shape}")

# Remove adult content
df = df[df['adult'] == False]
print(f"After removing adult         : {df.shape}")

# Remove empty titles
df = df[df['title'].notna() & (df['title'].str.strip() != '')]
print(f"After removing empty titles  : {df.shape}")

# Remove empty overview
df = df[df['overview'].notna() & (df['overview'].str.strip() != '')]
print(f"After removing no overview   : {df.shape}")

# Keep only released movies
df = df[df['status'] == 'Released']
print(f"After keeping released only  : {df.shape}")

# Remove movies with very few votes ← KEY CHANGE!
df = df[df['vote_count'] >= 50]
print(f"After removing low vote count: {df.shape}")

# Remove short films
df = df[(df['runtime'] >= 60) | (df['runtime'].isna())]

print(f"After removing short films   : {df.shape}")

# Remove movies with no genres
df = df[df['genres'].notna() & (df['genres'].str.strip() != '')]
print(f"After removing no genres     : {df.shape}")

df.reset_index(drop=True, inplace=True)
print(f"\n Final shape: {df.shape}")
print(f"Memory: {round(df.memory_usage(deep=True).sum()/1024**2,1)} MB")

Shape before filtering: (1385673, 15)
After removing adult         : (1247919, 15)
After removing empty titles  : (1247899, 15)
After removing no overview   : (962302, 15)
After keeping released only  : (926588, 15)
After removing low vote count: (27856, 15)
After removing short films   : (26498, 15)
After removing no genres     : (26490, 15)

 Final shape: (26490, 15)
Memory: 21.7 MB


In [49]:
C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.75)
# ↑ Changed from 0.70 → 0.75
#   Stricter threshold — only top 25%
#   most voted movies rank highly

print(f"Average rating (C) : {C:.2f}")
print(f"Min votes req. (m) : {m:.0f}")

def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    return round((v/(v+m)*R) + (m/(v+m)*C), 3)

df['weighted_rating'] = df.apply(weighted_rating, axis=1)

print("\nTop 10 by weighted rating:")
print(df.nlargest(10, 'weighted_rating')[
    ['title', 'vote_average', 'vote_count', 'weighted_rating']
].to_string(index=False))

Average rating (C) : 6.39
Min votes req. (m) : 470

Top 10 by weighted rating:
                   title  vote_average  vote_count  weighted_rating
The Shawshank Redemption         8.702       24649            8.659
           The Godfather         8.707       18677            8.650
        Schindler's List         8.573       14594            8.505
   The Godfather Part II         8.591       11293            8.503
         The Dark Knight         8.512       30619            8.480
           Spirited Away         8.539       14913            8.473
                Parasite         8.515       16430            8.456
            Pulp Fiction         8.488       25893            8.451
          The Green Mile         8.507       15937            8.446
            Forrest Gump         8.477       25409            8.439


In [50]:
def clean_genres_p2(text):
    try:
        if pd.isna(text) or str(text).strip() == '':
            return []
        return [
            g.strip().replace(" ", "")
            for g in str(text).split(',')
            if g.strip()
        ]
    except:
        return []

def clean_keywords_p2(text):
    try:
        if pd.isna(text) or str(text).strip() == '':
            return []
        return [
            k.strip().replace(" ", "")
            for k in str(text).split(',')
            if k.strip()
        ][:10]
    except:
        return []

def clean_title(title):
    try:
        title = str(title).lower()
        title = re.sub(r'[^a-z0-9\s]', '', title)
        return [w for w in title.split() if len(w) > 1]
    except:
        return []

def create_better_tags(row):
    tags = []

    title_words = clean_title(str(row['title']))
    tags       += title_words * 4

    genres  = clean_genres_p2(str(row['genres']))
    tags   += genres * 3

    keywords = clean_keywords_p2(str(row['keywords']))
    tags    += keywords * 2

    overview = str(row['overview']) if pd.notna(
        row['overview']) else ''
    overview_words = [
        w for w in overview.lower().split()[:50]
        if len(w) > 3
    ]
    tags += overview_words

    try:
        year   = int(str(row['release_date'])[:4])
        decade = f"{(year//10)*10}s"
        tags.append(decade)
    except:
        pass

    lang = str(row['original_language'])
    if lang and lang != 'nan':
        tags.append(f"lang{lang}")

    return " ".join(tags).lower()

print("Building tags... (~2 minutes)")
df['tags']          = df.apply(create_better_tags, axis=1)
df['genres_list']   = df['genres'].apply(clean_genres_p2)
df['keywords_list'] = df['keywords'].apply(clean_keywords_p2)

print(f"Done!")
print(f"Null tags  : {df['tags'].isna().sum()}")
print(f"Empty tags : {(df['tags'] == '').sum()}")
print(f"\nInception tags:")
inception = df[df['title'] == 'Inception']
if not inception.empty:
    print(inception['tags'].values[0][:300])

Building tags... (~2 minutes)
Done!
Null tags  : 0
Empty tags : 0

Inception tags:
inception inception inception inception action sciencefiction adventure action sciencefiction adventure action sciencefiction adventure rescue mission dream airplane paris france virtualreality kidnapping philosophy spy rescue mission dream airplane paris france virtualreality kidnapping philosophy 


In [51]:
# Check all columns exist before saving
print("Current columns:", df.columns.tolist())

final_columns = [
    'id', 'title', 'tags', 'genres_list',
    'keywords_list', 'weighted_rating', 'vote_count',
    'release_date', 'original_language',
    'poster_path', 'overview', 'imdb_id'
]

# Verify all columns exist
missing = [c for c in final_columns if c not in df.columns]
if missing:
    print(f"Missing columns: {missing}")
else:
    df_final = df[final_columns].copy()
    df_final.rename(columns={'id': 'movie_id'}, inplace=True)
    df_final.reset_index(drop=True, inplace=True)
    df_final.to_csv('data/tmdb_phase2_features.csv', index=False)
    print(f" Saved! Shape: {df_final.shape}")

Current columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'runtime', 'adult', 'imdb_id', 'original_language', 'overview', 'popularity', 'poster_path', 'genres', 'keywords', 'weighted_rating', 'tags', 'genres_list', 'keywords_list']
 Saved! Shape: (26490, 12)


In [52]:
import faiss
import pickle
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df = pd.read_csv('data/tmdb_phase2_features.csv')
df['genres_list'] = df['genres_list'].apply(
    lambda x: ast.literal_eval(x)
    if isinstance(x, str) else []
)
df['tags'] = df['tags'].fillna('')
print(f"Loaded! Shape: {df.shape}")

# TF-IDF
print("\nBuilding TF-IDF...")
start = time.time()
tfidf = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    min_df=2,
    max_df=0.85,
    sublinear_tf=True,
    ngram_range=(1, 2)
)
tfidf_matrix = tfidf.fit_transform(df['tags'])
print(f"TF-IDF : {tfidf_matrix.shape} | {round(time.time()-start,1)}s")

# SVD
print("\nBuilding SVD...")
start         = time.time()
svd           = TruncatedSVD(n_components=500, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)
print(f"SVD    : {tfidf_reduced.shape}")
print(f"Variance: {svd.explained_variance_ratio_.sum():.1%}")
print(f"Time   : {round(time.time()-start,1)}s")

# Normalize
print("\nNormalizing...")
tfidf_reduced_f32 = tfidf_reduced.astype('float32')
del tfidf_reduced

BATCH = 10000
for i in range(0, len(tfidf_reduced_f32), BATCH):
    batch                        = tfidf_reduced_f32[i:i+BATCH]
    norms                        = np.linalg.norm(batch, axis=1, keepdims=True)
    norms[norms == 0]            = 1
    tfidf_reduced_f32[i:i+BATCH] = batch / norms

tfidf_normalized = tfidf_reduced_f32
print(f"Shape  : {tfidf_normalized.shape}")
print(f"Norm   : {np.linalg.norm(tfidf_normalized[0]):.4f}")

# FAISS
print("\nBuilding FAISS...")
start     = time.time()
dimension = tfidf_normalized.shape[1]
index     = faiss.IndexFlatIP(dimension)
index.add(tfidf_normalized)
print(f"FAISS! Vectors:{index.ntotal:,} | {round(time.time()-start,1)}s")

Loaded! Shape: (26490, 12)

Building TF-IDF...
TF-IDF : (26490, 20000) | 9.4s

Building SVD...
SVD    : (26490, 500)
Variance: 28.1%
Time   : 26.8s

Normalizing...
Shape  : (26490, 500)
Norm   : 1.0000

Building FAISS...
FAISS! Vectors:26,490 | 0.1s


In [53]:
def recommend_p2(movie_title, n=10,
                 min_votes=500, language='all'):

    matches = df[df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        search_term = movie_title[:4].lower()
        suggestions = df[
            df['title'].str.lower().str.contains(search_term)
        ]['title'].head(5).tolist()
        print(f"'{movie_title}' not found!")
        print(f"Did you mean: {suggestions}")
        return None

    movie_idx    = matches.index[0]
    query_vector = tfidf_normalized[movie_idx:movie_idx+1]

    distances, indices_out = index.search(query_vector, 100)

    similar_indices   = indices_out[0][1:]
    similar_distances = distances[0][1:]

    results = df.iloc[similar_indices][
        ['movie_id', 'title', 'weighted_rating',
         'vote_count', 'release_date',
         'genres_list', 'original_language',
         'poster_path']
    ].copy()

    results['similarity'] = similar_distances.round(3)

    
    results = results[results['vote_count'] >= min_votes]

   
    if language != 'all':
        results = results[
            results['original_language'] == language
        ]

    max_votes = results['vote_count'].max()
    if max_votes > 0:
        results['popularity_score'] = (
            results['vote_count'] / max_votes * 0.3
        )
        
        results['final_score'] = (
            results['similarity'] * 0.7 +
            results['popularity_score']
        )
        
    else:
        results['final_score'] = results['similarity']

    results = results.sort_values(
        'final_score', ascending=False
    )

    results = results.head(n)
    results = results.reset_index(drop=True)
    results.index = results.index + 1

    return results


test_movies = ["Inception", "Avatar",
               "The Dark Knight", "Titanic"]

for movie in test_movies:
    results = recommend_p2(movie, n=5, min_votes=500)
    if results is not None:
        print(f" Top 5 for '{movie}':")
        for idx, row in results.iterrows():
            print(f"  {idx}. {str(row['title']):45s}"
                  f" votes:{int(row['vote_count']):5,}"
                  f" sim:{row['similarity']:.3f}"
                  f" final:{row['final_score']:.3f}")
        print()

 Top 5 for 'Inception':
  1. Avengers: Infinity War                        votes:27,713 sim:0.418 final:0.593
  2. Guardians of the Galaxy                       votes:26,638 sim:0.398 final:0.567
  3. The Empire Strikes Back                       votes:15,773 sim:0.534 final:0.545
  4. Iron Man                                      votes:24,874 sim:0.390 final:0.542
  5. The Hunger Games: Catching Fire               votes:16,182 sim:0.491 final:0.519

 Top 5 for 'Avatar':
  1. The Hunger Games                              votes:20,636 sim:0.427 final:0.599
  2. Fantastic Four                                votes:8,765 sim:0.666 final:0.594
  3. The Lord of the Rings: The Two Towers         votes:20,274 sim:0.426 final:0.593
  4. Divergent                                     votes:12,118 sim:0.496 final:0.523
  5. Dark Phoenix                                  votes:5,941 sim:0.601 final:0.507

 Top 5 for 'The Dark Knight':
  1. The Dark Knight Rises                         votes:21,335 s

In [54]:
import scipy.sparse
import os

faiss.write_index(index, 'data/faiss_index.bin')
np.save('data/tfidf_normalized.npy', tfidf_normalized)
scipy.sparse.save_npz('data/tfidf_sparse.npz', tfidf_matrix)

with open('data/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
with open('data/tfidf_vectorizer_p2.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

df.to_csv('data/tmdb_phase2_features.csv', index=False)

print(" All files saved!")
print("\n File sizes:")
files = [
    'data/faiss_index.bin',
    'data/tfidf_normalized.npy',
    'data/tfidf_sparse.npz',
    'data/svd_model.pkl',
    'data/tfidf_vectorizer_p2.pkl',
    'data/tmdb_phase2_features.csv'
]
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024**2
        print(f"  {f:40s} → {size:.1f} MB")

 All files saved!

 File sizes:
  data/faiss_index.bin                     → 50.5 MB
  data/tfidf_normalized.npy                → 50.5 MB
  data/tfidf_sparse.npz                    → 7.9 MB
  data/svd_model.pkl                       → 76.3 MB
  data/tfidf_vectorizer_p2.pkl             → 0.8 MB
  data/tmdb_phase2_features.csv            → 23.0 MB
